In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import libraries
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# load data
path = '/content/drive/MyDrive/Internship_Uzcosmos/datasets/Sentinel2_Cloud_Snow_Detection_Dataset.tif'

with rasterio.open(path) as src:
  print(f"All information: {src.profile}")

  data = src.read().astype(float)

  print(f"Shape: {data.shape}")
  print(f"Min: {data.min():.4f}")
  print(f"Max: {data.max():.4f}")

  input = data[0:5].astype('float32')
  output_scl = data[5].astype('uint8')

All information: {'driver': 'GTiff', 'dtype': 'uint16', 'nodata': None, 'width': 4222, 'height': 2825, 'count': 6, 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 42N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",69],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32642"]]'), 'transform': Affine(20.0, 0.0, 541720.0,
       0.0, -20.0, 4595360.0), 'blockxsize': 256, 'blockysize': 256, 'tiled': True, 'compress': 'lzw', 'interleave': 'pixel'}
Shape: (6, 2825, 4222)
Min: 0.0000
Max: 23512.0000


In [ ]:
# Normalize input data and show min and max
normalized_data = input / 10000.0

data_names = ['B2', 'B3', 'B4', 'B8', 'B11']
for i in range(normalized_data.shape[0]):
  band = normalized_data[i]
  print(f"Band {i} ({data_names[i]}): min={band.min():.4f}, max={band.max():.4f}")

B2 = normalized_data[0]
B3 = normalized_data[1]
B4 = normalized_data[2]
B8 = normalized_data[3]
B11 = normalized_data[4]

Band 0 (B2): min=0.0000, max=2.3512
Band 1 (B3): min=0.0000, max=2.1304
Band 2 (B4): min=0.0000, max=1.9624
Band 3 (B8): min=0.0000, max=1.7720
Band 4 (B11): min=0.0000, max=1.5774


In [ ]:
# check output scl
print("SCL values and pixel counts: ")
unique, counts = np.unique(output_scl, return_counts=True)
total = output_scl.size

scl_meaning = {
    0: 'No data',
    1: 'Saturated/defective',
    2: 'Dark area',
    3: 'Cloud shadow',
    4: 'Vegatation',
    5: 'Bare soil',
    6: 'Water',
    7: 'Unclassified',
    8: 'Cloud medium prob',
    9: 'Cloud high prob',
    10: 'Thin cirrus',
    11: 'Snow/Ice'
}

for val, count in zip(unique, counts):
  meaning = scl_meaning.get(val, 'Unknown')
  percentage = (count / total) * 100
  bar = '█' * int(percentage)
  print(f"SCL {val:2d} ({meaning: <22}): {count: > 8,} pixels {percentage:5.1f}% {bar}")

SCL values and pixel counts: 
SCL  0 (No data               ):  8,232,050 pixels  69.0% █████████████████████████████████████████████████████████████████████
SCL  2 (Dark area             ):   16,886 pixels   0.1% 
SCL  3 (Cloud shadow          ):  196,786 pixels   1.6% █
SCL  4 (Vegatation            ):  963,740 pixels   8.1% ████████
SCL  5 (Bare soil             ):  1,120,915 pixels   9.4% █████████
SCL  6 (Water                 ):   11,437 pixels   0.1% 
SCL  7 (Unclassified          ):   19,577 pixels   0.2% 
SCL  8 (Cloud medium prob     ):  369,872 pixels   3.1% ███
SCL  9 (Cloud high prob       ):  791,178 pixels   6.6% ██████
SCL 10 (Thin cirrus           ):      339 pixels   0.0% 
SCL 11 (Snow/Ice              ):  204,370 pixels   1.7% █


In [ ]:
# make output like ready mask
ready_mask = np.full(output_scl.shape, 255, dtype='uint8')

# this is clear
ready_mask[(output_scl == 4) | (output_scl == 5)] = 0

# this is cloud
ready_mask[(output_scl == 8) | (output_scl == 9)] = 1

print(ready_mask.shape)

unique_mask, count_mask = np.unique(ready_mask, return_counts=True)
print("final mask =>")
for val, count in zip(unique_mask, count_mask):
  label = "clear" if val == 0 else "cloud" if val == 1 else "ignored"
  print(f"{label:<8} (Value {val:3d}): {count:>10,} px")

(2825, 4222)
final mask =>
clear    (Value   0):  2,084,655 px
cloud    (Value   1):  1,161,050 px
ignored  (Value 255):  8,681,445 px


In [ ]:
#  Flatten spatial dimensions into a table of pixels
# (5, H, W) → (H, W, 5) → (H*W, 5)
X_all = normalized_data.transpose(1, 2, 0).reshape(-1, 5)

# and also, let's do flatten for mask to (H*W, )
y_all =ready_mask.reshape(-1)

# check there is how many row and column
print(f"Total pixels: {X_all.shape[0]:,}")
print(f"Features per pixel: {X_all.shape[1]}")

# remove invalid values, like ignored
valid_mask = y_all != 255
X = X_all[valid_mask]
y = y_all[valid_mask]

print(f"\nValid pixels for training: {X.shape[0]:,} px")
print(f"Sum of clear pixels(0): {(y == 0).sum():,} px")
print(f"Sum of cloud pixels(1): {(y == 1).sum():,} px")

# split data for train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# check
print(f"\nTraining samples: {X_train.shape[0]:,}")
print(f"Test samples: {X_test.shape[0]:,}")

Total pixels: 11,927,150
Features per pixel: 5

Valid pixels for training: 3,245,705 px
Sum of clear pixels(0): 2,084,655 px
Sum of cloud pixels(1): 1,161,050 px

Training samples: 2,596,564
Test samples: 649,141


In [ ]:
# choose model and train it
model = RandomForestClassifier(
    n_estimators=100, max_depth=10, n_jobs=-1, random_state=42, class_weight='balanced'
)

# train model
print("Training model")
model.fit(X_train, y_train)
print("Finished training")

Training model


In [ ]:
# test model
y_predict = model.predict(X_test)

print("Classificatio Report: ", classification_report(y_test, y_predict, target_names=['Clear (0)', 'Cloud (1)']))

# check that which band matters most
print("\nBand importance for cloud detection")
for name, importance in zip(data_names, model.feature_importances_):
  bar = '█' * int(importance * 50)
  print(f"  {name}: {importance:.3f} {bar}")